<a href="https://colab.research.google.com/github/Jules-Vatel/SSI_SPRING/blob/main/SSI_SPRING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

df = pd.read_csv("/content/Offer-Westort_April 28, 2026_08.08.csv")

df = df[-df['Treat_Party'].str.contains('ImportId',na=False)].copy()
df['Feeling_ThermR'] = pd.to_numeric(df['Q-FeelingThermR_1'], errors='coerce')
df['Feeling_ThermD'] = pd.to_numeric(df['Q-FeelingThermD_1'], errors='coerce')



In [31]:
def seven_pt_pid(row):
  party = row['Q-PartyAffiliation']
  strength = row['Q-PolarAffiliation']
  lean = row['Q-MiddleAffiliation']

  if party == 'Democrat':
      return 1 if strength == 'Strong' else 2
  elif party == 'Republican':
      return 7 if strength == 'Strong' else 6
  elif party in ['Independent', 'No Preference', 'Other Party (Please Specify)']:
      if lean == 'Closer to Democratic Party':
          return 3
      elif lean == 'Closer to Republican Party':
          return 5
      elif lean == 'Neither':
          return 4
  return np.nan

df['PA']=df.apply(seven_pt_pid,axis=1)

In [28]:
def get_party(row):
    pa = row['Q-PartyAffiliation']
    if pa == 'Democrat':
        return 'Democrat'
    elif pa == 'Republican':
        return 'Republican'
    elif pa in ['Independent', 'No Preference', 'Other Party (Please Specify)']:
        ma = row.get('Q-MiddleAffiliation', np.nan)
        if ma == 'Closer to Democratic Party':
            return 'Democrat'
        elif ma == 'Closer to Republican Party':
            return 'Republican'
    return np.nan

df['Party'] = df.apply(get_party, axis=1)


In [29]:
df['ThermInP']  = np.where(df['Party'] == 'Democrat', df['Feeling_ThermD'], df['Feeling_ThermR'])
df['ThermOutP'] = np.where(df['Party'] == 'Democrat', df['Feeling_ThermR'], df['Feeling_ThermD'])
df['APpre'] = (df['ThermInP'] - df['ThermOutP']).abs()

comfort_map = {
    'Very uncomfortable': 1, 'Uncomfortable': 2, 'Somewhat uncomfortable': 3,
    'Neither comfortable nor uncomfortable': 4, 'Somewhat comfortable': 5,
    'Comfortable': 6, 'Very comfortable': 7, 'Very Comfortable': 7
}

for col in ['Q-InLawDist', 'Q-CloseFriendDist', 'Q-Social Distance 3']:
    df[col + '_n'] = df[col].map(comfort_map)
df['APpost'] = 8 - df[['Q-InLawDist_n', 'Q-CloseFriendDist_n',
                        'Q-Social Distance 3_n']].mean(axis=1)
reelect_map = {
    'Very unlikely': 1, 'Unlikely': 2, 'Somewhat unlikely': 3,
    'Neither likely nor unlikely': 4, 'Somewhat likely': 5,
    'Likely': 6, 'Very likely': 7
}
df['TB'] = df['Q-ReElection'].map(reelect_map)



In [40]:
df['Treat'] = (df['Treat_Party'] != 'Control').astype(int)
df['Pin']   = (df['Treat_Party'] == 'InParty').astype(int)
df['FE']    = (df['Treat_Frame'] == 'Electoral').astype(int)
df['FD']    = (df['Treat_Frame'] == 'Democracy').astype(int)

df['FE_x_Pin'] = df['FE'] * df['Pin']
df['FD_x_Pin'] = df['FD'] * df['Pin']

analysis_vars = ['APpost','TB','APpre','PA']
full = df.dropna(subset=analysis_vars).copy()
treated = full[full['Treat']==1].copy()

def run_ols(y,x_vars,data):
  Y=data[y]
  X=sm.add_constant(data[x_vars])
  return sm.OLS(Y,X).fit(cov_type='HC1')



In [43]:
AP_1 = run_ols('APpost',['Treat','APpre','PA'],full)
TB_1 = run_ols('TB',['Treat','APpre','PA'],full)
AP_2 = run_ols('APpost',['Treat','APpre','PA'],treated)
TB_2 = run_ols('TB',['Pin','APpre','PA'],treated)
AP_3 = run_ols('APpost',['FE','FD','Pin','APpre','PA'],treated)
TB_3 = run_ols('TB',['FE','FD','Pin','APpre','PA'], treated)
AP_4 = run_ols('APpost',['FE','FD','Pin','FE_x_Pin','FD_x_Pin','APpre','PA'],treated)
TB_4 = run_ols('TB',['FE','FD','Pin','FE_x_Pin','FD_x_Pin','APpre','PA'],treated)

ftest_AP3 = AP_3.f_test("FE = FD")
ftest_TB3 = TB_3.f_test("FE = FD")
ftest_AP4 = AP_4.f_test("FE_x_Pin = 0, FD_x_Pin = 0")
ftest_TB4 = TB_4.f_test("FE_x_Pin = 0, FD_x_Pin = 0")

In [42]:
results = {
    'rq1': {'APpost': rq1_ap, 'TB': rq1_tb},
    'rq2': {'APpost': rq2_ap, 'TB': rq2_tb},
    'rq3': {'APpost': rq3_ap, 'TB': rq3_tb,
             'ftest_FE_eq_FD_ap': ftest_rq3_ap,
             'ftest_FE_eq_FD_tb': ftest_rq3_tb},
    'rq4': {'APpost': rq4_ap, 'TB': rq4_tb,
             'ftest_joint_interactions_ap': ftest_rq4_ap,
             'ftest_joint_interactions_tb': ftest_rq4_tb},
    'samples': {'full_n': len(full), 'treated_n': len(treated)},
    'data': {'full': full, 'treated': treated}
}

NameError: name 'rq1_ap' is not defined